# limit_impact_v01：批量评估 D+1 涨跌停/停牌对拟合篮子的影响

本 notebook 参考 `correlation_v04.ipynb` 的任务结构，对观察区间内的每个构建日 D 独立执行完整流程：

1. 使用 D 日权重和收盘价构建 basket1；
2. 使用迅投历史数据识别 D+1 09:31 涨停、跌停、停牌与人工禁买；
3. basket3 在全部指数成分股中排除不可购买股票后重新拟合；
4. 以 D+1 09:31 close 计算 basket1 和 basket3 的偏差；
5. Task11 汇总为 `2 * len(construction_date_ls)` 行的大表，并使用 Plotly 绘制三张交互折线图。

当 `construction_date_ls` 恰好包含两个日期时，两者被解释为观察区间的开始和结束，并使用迅投交易日历展开为闭区间内的全部交易日。本流程不使用同花顺数据源。


## Task0：导入与指数配置


In [15]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display
from xtquant import xtdata

xtdata.enable_hello = False
xtdata.reconnect(port=58610)

from utils import (
    LimitImpactDateRun,
    LimitImpactPipelineConfig,
    combine_task11_summaries,
    expand_construction_date_list,
    filter_available_date_runs,
    preflight_transition_daily_data,
    gogoal_query,
    read_daily_data,
)

PROJECT_ROOT = Path.cwd().resolve()

INDEX_CODE = "000300"
INDEX_META = {
    "000016": {"name": "上证50", "xt": "000016.SH"},
    "000300": {"name": "沪深300", "xt": "000300.SH"},
    "000905": {"name": "中证500", "xt": "000905.SH"},
    "000852": {"name": "中证1000", "xt": "000852.SH"},
    "000688": {"name": "科创50", "xt": "000688.SH"},
}
if INDEX_CODE not in INDEX_META:
    raise ValueError(f"INDEX_META 未配置指数 {INDEX_CODE}")

index_name = INDEX_META[INDEX_CODE]["name"]
xt_index_code = INDEX_META[INDEX_CODE]["xt"]
print("项目目录:", PROJECT_ROOT)
print("指数:", INDEX_CODE, index_name)


项目目录: E:\Codex\系统\Stock-Index-Fitting
指数: 000300 沪深300


## Task1：运行参数

`construction_date_ls` 恰好有两个值时，分别为观察开始日和结束日；代码使用迅投上交所日历取闭区间内的全部交易日。边界可以是非交易日，但只有交易日会进入实际运行列表。若列表长度不是 2，则仍按显式构建日列表处理。


In [16]:
# 恰好两个日期时：[观察开始日, 观察结束日]。
construction_date_ls = ["20260701", "20260731"]

# 人工指定的不可购买股票；可为空，对所有日期共用。
manual_unavailable_codes = ["000001"]
# manual_unavailable_codes = []
# manual_unavailable_codes = ["600000", "000001.SZ"]

target_stock_value = 4_500_000.0
rule_file_path = "security_buy_rules.csv"

# 与 correlation_v04 一致的风险模型和优化参数，使用优化后的 Pareto 搜索。
risk_matrix_mode = "correlation"  # "covariance" 或 "correlation"
risk_lookback_days = 5
risk_half_life_days = 3.0
risk_price_col = "lastPrice"

pareto_risk_candidate_count = 10
pareto_amount_candidate_count = 10
pareto_beam_width = 20
pareto_max_rounds = 50
pareto_stale_rounds_to_stop = 3
pareto_legal_neighbor_steps = 3

allow_over_budget = True
max_over_budget_ratio = 1.005

# 迅投原始三角 tick 的既有 NAS 路径。
source_tick_root = Path(r"Z:\高频行情迅投\ticks")
price_validation_tolerance = 0.0001
limit_price_tolerance = 1e-6
include_suspended_as_unavailable = True

construction_date_input_ls = list(construction_date_ls)
construction_date_ls = expand_construction_date_list(
    xtdata,
    construction_date_input_ls,
)

print("输入日期/区间:", construction_date_input_ls)
print("迅投日历展开后的构建日:", construction_date_ls)
print("实际构建日数量:", len(construction_date_ls))


输入日期/区间: ['20260701', '20260731']
迅投日历展开后的构建日: ['20260701', '20260702', '20260703', '20260706', '20260707', '20260708', '20260709', '20260710', '20260713', '20260714', '20260715', '20260716', '20260717', '20260720', '20260721', '20260722', '20260723', '20260724', '20260727', '20260728', '20260729', '20260730', '20260731']
实际构建日数量: 23


## Task2：初始化批量运行与日期隔离目录

每个构建日在 `date_runs/<construction_date>/` 下保留原有的输入、状态、篮子和报表；批量 Task11 结果保存在本次运行根目录的 `04_reports/`。


In [17]:
started_at = datetime.now().astimezone()
import_time = started_at.strftime("%Y%m%d-%H%M%S")
batch_run_dir = (
    PROJECT_ROOT
    / "data"
    / INDEX_CODE
    / f"{import_time}_limit_impact_v01_batch"
)
batch_reports_dir = batch_run_dir / "04_reports"
batch_reports_dir.mkdir(parents=True, exist_ok=True)

pipeline_config = LimitImpactPipelineConfig(
    index_code=INDEX_CODE,
    index_name=index_name,
    xt_index_code=xt_index_code,
    target_stock_value=target_stock_value,
    rule_file_path=rule_file_path,
    manual_unavailable_codes=tuple(manual_unavailable_codes),
    risk_matrix_mode=risk_matrix_mode,
    risk_lookback_days=risk_lookback_days,
    risk_half_life_days=risk_half_life_days,
    risk_price_col=risk_price_col,
    pareto_risk_candidate_count=pareto_risk_candidate_count,
    pareto_amount_candidate_count=pareto_amount_candidate_count,
    pareto_beam_width=pareto_beam_width,
    pareto_max_rounds=pareto_max_rounds,
    pareto_stale_rounds_to_stop=pareto_stale_rounds_to_stop,
    pareto_legal_neighbor_steps=pareto_legal_neighbor_steps,
    allow_over_budget=allow_over_budget,
    max_over_budget_ratio=max_over_budget_ratio,
    source_tick_root=source_tick_root,
    price_validation_tolerance=price_validation_tolerance,
    limit_price_tolerance=limit_price_tolerance,
    include_suspended_as_unavailable=(
        include_suspended_as_unavailable
    ),
)

date_runs = [
    LimitImpactDateRun(
        construction_date=construction_date,
        config=pipeline_config,
        project_root=PROJECT_ROOT,
        batch_run_dir=batch_run_dir,
        import_time=import_time,
        xtdata_client=xtdata,
        gogoal_query_fn=gogoal_query,
        daily_loader=read_daily_data,
    )
    for construction_date in construction_date_ls
]

print("日期数量:", len(date_runs))
print("构建日 -> D+1:")
for run in date_runs:
    print(" ", run.construction_date, "->", run.evaluation_date)
print("批量输出目录:", batch_run_dir)


日期数量: 23
构建日 -> D+1:
  20260701 -> 20260702
  20260702 -> 20260703
  20260703 -> 20260706
  20260706 -> 20260707
  20260707 -> 20260708
  20260708 -> 20260709
  20260709 -> 20260710
  20260710 -> 20260713
  20260713 -> 20260714
  20260714 -> 20260715
  20260715 -> 20260716
  20260716 -> 20260717
  20260717 -> 20260720
  20260720 -> 20260721
  20260721 -> 20260722
  20260722 -> 20260723
  20260723 -> 20260724
  20260724 -> 20260727
  20260727 -> 20260728
  20260728 -> 20260729
  20260729 -> 20260730
  20260730 -> 20260731
  20260731 -> 20260803
批量输出目录: E:\Codex\系统\Stock-Index-Fitting\data\000300\20260807-165523_limit_impact_v01_batch


## Task3：预检日行情/minute cache，再逐日读取权重与交易规则

任一交易日 T 的 NAS 日行情缺失或为空时，同时跳过所有端点触及 T 的区间，即 `前一交易日 -> T` 与 `T -> 下一交易日`。处理方式与 unavailable minute cache 一致：记录、过滤并继续其余区间，不中断批量运行。


In [18]:
# 1. 先处理已存在的 unavailable minute-cache marker。
unavailable_trade_dates = set()
for run in date_runs:
    if set(run.transition_dates) & unavailable_trade_dates:
        continue
    unavailable_trade_dates.update(
        run.existing_unavailable_transition_dates()
    )

date_runs = filter_available_date_runs(
    date_runs,
    unavailable_trade_dates,
)
if not date_runs:
    raise RuntimeError("没有可计算的 D->D+1 区间。")

# 2. 一次性读取每个不同 D/D+1 日期的 NAS 日行情。
#    若 T 缺失，filter_available_date_runs 会同时删除两侧区间。
daily_preflight_input_runs = list(date_runs)
(
    date_runs,
    nas_daily_by_date,
    df_nas_daily_preflight,
) = preflight_transition_daily_data(
    daily_preflight_input_runs,
    read_daily_data,
)
unavailable_nas_daily_dates = set(
    df_nas_daily_preflight.loc[
        ~df_nas_daily_preflight["is_available"],
        "trade_date",
    ]
)
unavailable_trade_dates.update(unavailable_nas_daily_dates)

available_transition_keys = {
    run.transition_dates for run in date_runs
}
df_skipped_nas_daily_transitions = pd.DataFrame(
    [
        {
            "construction_date": run.construction_date,
            "evaluation_date": run.evaluation_date,
            "touching_unavailable_dates": "|".join(
                sorted(
                    set(run.transition_dates)
                    & unavailable_nas_daily_dates
                )
            ),
            "reason": "nas_daily_missing_or_empty",
        }
        for run in daily_preflight_input_runs
        if run.transition_dates not in available_transition_keys
        and set(run.transition_dates) & unavailable_nas_daily_dates
    ],
    columns=[
        "construction_date",
        "evaluation_date",
        "touching_unavailable_dates",
        "reason",
    ],
)
df_nas_daily_preflight.to_csv(
    batch_reports_dir / "nas_daily_preflight.csv",
    index=False,
    encoding="utf-8-sig",
)
df_skipped_nas_daily_transitions.to_csv(
    batch_reports_dir / "skipped_nas_daily_transitions.csv",
    index=False,
    encoding="utf-8-sig",
)

construction_date_ls = [
    run.construction_date for run in date_runs
]
if not date_runs:
    raise RuntimeError(
        "NAS 日行情预检后没有可计算的 D->D+1 区间。"
    )
print("NAS 日行情缺失日:", sorted(unavailable_nas_daily_dates))
print(
    "NAS 日行情导致跳过的区间数:",
    len(df_skipped_nas_daily_transitions),
)

# 3. 仅对日行情可用的区间读取权重和交易规则。
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task3 "
        f"{run.construction_date}"
    )
    run.load_inputs()

# 4. 复用现有 minute-cache 预检，同样删除触及不可用日的区间。
for run in date_runs:
    if set(run.transition_dates) & unavailable_trade_dates:
        continue
    unavailable_trade_dates.update(
        run.preflight_transition_minute_caches()
    )

date_runs = filter_available_date_runs(
    date_runs,
    unavailable_trade_dates,
)
construction_date_ls = [
    run.construction_date for run in date_runs
]
if not date_runs:
    raise RuntimeError("预检后没有可计算的 D->D+1 区间。")
print("预检后有效区间数:", len(date_runs))

if not df_nas_daily_preflight["is_available"].all():
    display(
        df_nas_daily_preflight.loc[
            ~df_nas_daily_preflight["is_available"]
        ]
    )
    display(df_skipped_nas_daily_transitions)

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "evaluation_date": run.evaluation_date,
                "component_count": len(run.stock_codes),
                "raw_weight_sum_pct": run.df_index_weights[
                    "raw_weight_pct"
                ].sum(),
            }
            for run in date_runs
        ]
    )
)


[UNAVAILABLE CACHE HIT] transition preflight 20260720: reason=source_file_missing; details={'source_files': ['Z:\\高频行情迅投\\ticks\\SH\\2026\\07\\20260720_tick_sh_kcb.pkl', 'Z:\\高频行情迅投\\ticks\\SH\\2026\\07\\20260720_tick_sh_zb.pkl', 'Z:\\高频行情迅投\\ticks\\SZ\\2026\\07\\20260720_tick_sz_cyb.pkl', 'Z:\\高频行情迅投\\ticks\\SZ\\2026\\07\\20260720_tick_sz_zb.pkl'], 'message': "D+1 opening cache 20260717 20260720 requires 4 missing source file(s): ['Z:\\\\高频行情迅投\\\\ticks\\\\SH\\\\2026\\\\07\\\\20260720_tick_sh_kcb.pkl', 'Z:\\\\高频行情迅投\\\\ticks\\\\SH\\\\2026\\\\07\\\\20260720_tick_sh_zb.pkl', 'Z:\\\\高频行情迅投\\\\ticks\\\\SZ\\\\2026\\\\07\\\\20260720_tick_sz_cyb.pkl', 'Z:\\\\高频行情迅投\\\\ticks\\\\SZ\\\\2026\\\\07\\\\20260720_tick_sz_zb.pkl']"}
[UNAVAILABLE CACHE HIT] transition preflight 20260729: reason=source_file_missing; details={'source_files': ['Z:\\高频行情迅投\\ticks\\SZ\\2026\\07\\20260729_tick_sz_cyb.pkl', 'Z:\\高频行情迅投\\ticks\\SZ\\2026\\07\\20260729_tick_sz_zb.pkl'], 'message': "tracking stock minute cache 2

,construction_date,evaluation_date,component_count,raw_weight_sum_pct
0,20260701,20260702,300,100.0
1,20260702,20260703,300,100.0
2,20260703,20260706,300,100.0
3,20260706,20260707,300,100.0
4,20260707,20260708,300,100.0
5,20260708,20260709,300,100.0
6,20260709,20260710,300,100.0
7,20260710,20260713,300,100.0
8,20260713,20260714,300,100.0
9,20260714,20260715,300,100.0


## Task4：逐日读取 D 日指数与成分股收盘价

Go-Goal 为构建价主源，迅投指数日线和 NAS 股票日线用于交叉校验。Task3 已读取并缓存可用日行情，本任务直接复用，不重复读盘。


In [19]:
price_ready_runs = []
late_unavailable_nas_daily_dates = set()
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task4 "
        f"{run.construction_date}"
    )
    nas_daily = nas_daily_by_date.get(run.construction_date)
    if run.load_construction_prices(nas_daily=nas_daily):
        price_ready_runs.append(run)
    else:
        late_unavailable_nas_daily_dates.add(
            run.construction_date
        )

# 即使预检后数据被外部删除，也按同一规则继续，不中断。
unavailable_trade_dates.update(
    late_unavailable_nas_daily_dates
)
date_runs = filter_available_date_runs(
    price_ready_runs,
    late_unavailable_nas_daily_dates,
)
construction_date_ls = [
    run.construction_date for run in date_runs
]
nas_daily_by_date.clear()
del nas_daily_by_date

if not date_runs:
    raise RuntimeError(
        "NAS 日行情过滤后没有可计算的 D->D+1 区间。"
    )

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "construction_index_close": (
                    run.construction_index_close
                ),
                "validated_stock_count": len(
                    run.df_market_snapshot
                ),
            }
            for run in date_runs
        ]
    )
)


[1/19] Task4 20260701


正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库
正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库


[2/19] Task4 20260702


正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库
正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库


[3/19] Task4 20260703


...

[18/19] Task4 20260730


正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库
正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库


[19/19] Task4 20260731


正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库
正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库


,construction_date,construction_index_close,validated_stock_count
0,20260701,4958.9773,300
1,20260702,4812.2957,300
2,20260703,4842.1737,300
3,20260706,4841.9980,300
4,20260707,4792.2624,300
5,20260708,4755.5338,300
6,20260709,4876.3125,300
7,20260710,4780.7867,300
8,20260713,4695.3830,300
9,20260714,4796.5020,300


## Task5：逐日构建理论组合与风险矩阵


In [20]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task5 "
        f"{run.construction_date}"
    )
    run.build_risk_model()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "requested_risk_dates": run.risk_dates,
                "used_risk_dates": run.used_risk_dates,
                "skipped_risk_dates": run.skipped_risk_dates,
                "risk_matrix_shape": run.risk_matrix.shape,
                "oas_shrinkage": run.risk_model.summary[
                    "oas_shrinkage"
                ],
            }
            for run in date_runs
        ]
    )
)


[1/19] Task5 20260701
[CACHE HIT] risk cache 20260625 20260625: basket_minute_wide_20260625_f2653c108899.pkl (3.99 MB)
[CACHE HIT] risk cache 20260626 20260626: basket_minute_wide_20260626_e075d94ff3c9.pkl (3.99 MB)
[CACHE HIT] risk cache 20260629 20260629: basket_minute_wide_20260629_2d4119158d06.pkl (3.99 MB)
[CACHE HIT] risk cache 20260630 20260630: basket_minute_wide_20260630_353f712e13f0.pkl (3.99 MB)
[CACHE HIT] risk cache 20260701 20260701: basket_minute_wide_20260701_bfeca5272447.pkl (3.99 MB)
[Risk fallback 20260629] 使用 lastClose 填充全天无有效 lastPrice 的股票，共 1 只：['688072.SH']
[Risk fallback 20260630] 使用 lastClose 填充全天无有效 lastPrice 的股票，共 1 只：['688072.SH']
[Risk fallback 20260701] 使用 lastClose 填充全天无有效 lastPrice 的股票，共 1 只：['688072.SH']
[2/19] Task5 20260702
[CACHE HIT] risk cache 20260626 20260626: basket_minute_wide_20260626_e075d94ff3c9.pkl (3.99 MB)
[CACHE HIT] risk cache 20260629 20260629: basket_minute_wide_20260629_2d4119158d06.pkl (3.99 MB)
[CACHE HIT] risk cache 20260630 20260

,construction_date,requested_risk_dates,used_risk_dates,skipped_risk_dates,risk_matrix_shape,oas_shrinkage
0,20260701,"[20260625, 20260626, 20260629, 20260630, 20260...","[20260625, 20260626, 20260629, 20260630, 20260...",[],"(300, 300)",0.029701
1,20260702,"[20260626, 20260629, 20260630, 20260701, 20260...","[20260626, 20260629, 20260630, 20260701, 20260...",[],"(300, 300)",0.024132
2,20260703,"[20260629, 20260630, 20260701, 20260702, 20260...","[20260629, 20260630, 20260701, 20260702, 20260...",[],"(300, 300)",0.020576
3,20260706,"[20260630, 20260701, 20260702, 20260703, 20260...","[20260630, 20260701, 20260702, 20260703, 20260...",[],"(300, 300)",0.018401
4,20260707,"[20260701, 20260702, 20260703, 20260706, 20260...","[20260701, 20260702, 20260703, 20260706, 20260...",[],"(300, 300)",0.016044
5,20260708,"[20260702, 20260703, 20260706, 20260707, 20260...","[20260702, 20260703, 20260706, 20260707, 20260...",[],"(300, 300)",0.012864
6,20260709,"[20260703, 20260706, 20260707, 20260708, 20260...","[20260703, 20260706, 20260707, 20260708, 20260...",[],"(300, 300)",0.013993
7,20260710,"[20260706, 20260707, 20260708, 20260709, 20260...","[20260706, 20260707, 20260708, 20260709, 20260...",[],"(300, 300)",0.015036
8,20260713,"[20260707, 20260708, 20260709, 20260710, 20260...","[20260707, 20260708, 20260709, 20260710, 20260...",[],"(300, 300)",0.013484
9,20260714,"[20260708, 20260709, 20260710, 20260713, 20260...","[20260708, 20260709, 20260710, 20260713, 20260...",[],"(300, 300)",0.010723


## Task6：按原流程逐日构建 basket1


In [21]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task6 "
        f"{run.construction_date}"
    )
    run.build_basket1()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "basket1_invested_amount": (
                    run.basket1_invested_amount
                ),
                "basket1_held_stock_count": int(
                    (run.basket1["target_qty"] > 0).sum()
                ),
            }
            for run in date_runs
        ]
    )
)


[1/19] Task6 20260701
[pareto basket1] round=1, raw=1,694, local=63, materialized=63, frontier=63, beam=20, new=63, best_amount=0.17424769, best_TE=555.572522%
[pareto basket1] round=5, raw=37,873, local=741, materialized=736, frontier=105, beam=20, new=105, best_amount=0.16816950, best_TE=500.710978%
[pareto basket1] round=10, raw=28,274, local=563, materialized=553, frontier=102, beam=20, new=101, best_amount=0.16370712, best_TE=482.737040%
[pareto basket1] round=15, raw=23,955, local=384, materialized=364, frontier=113, beam=20, new=107, best_amount=0.16130604, best_TE=468.558171%
[pareto basket1] round=20, raw=16,262, local=301, materialized=295, frontier=117, beam=20, new=86, best_amount=0.15892463, best_TE=461.518452%
[pareto basket1] round=25, raw=14,850, local=281, materialized=267, frontier=181, beam=20, new=97, best_amount=0.15803714, best_TE=452.984629%
[pareto basket1] round=30, raw=10,689, local=255, materialized=234, frontier=138, beam=20, new=96, best_amount=0.15795575, 

,construction_date,basket1_invested_amount,basket1_held_stock_count
0,20260701,4514954.90,274
1,20260702,4514452.07,278
2,20260703,4509257.47,281
3,20260706,4519512.45,281
4,20260707,4518819.08,277
5,20260708,4500072.13,277
6,20260709,4521931.72,282
7,20260710,4517055.81,282
8,20260713,4500084.89,281
9,20260714,4505582.12,281


## Task7：逐日读取 D+1 09:31 价格与迅投涨跌停/停牌状态


In [22]:
late_unavailable_dates = set()
status_ready_runs = []
for position, run in enumerate(date_runs, start=1):
    if set(run.transition_dates) & late_unavailable_dates:
        continue
    print(
        f"[{position}/{len(date_runs)}] Task7 "
        f"{run.construction_date} -> {run.evaluation_date}"
    )
    if run.load_d1_status():
        status_ready_runs.append(run)
    else:
        late_unavailable_dates.add(run.evaluation_date)

date_runs = filter_available_date_runs(
    status_ready_runs, late_unavailable_dates
)
construction_date_ls = [run.construction_date for run in date_runs]
if not date_runs:
    raise RuntimeError("D+1 状态检查后没有可计算区间。")

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "evaluation_date": run.evaluation_date,
                "unavailable_component_count": len(
                    run.status_unavailable_codes
                ),
                "manual_not_in_universe": (
                    run.manual_not_in_universe
                ),
            }
            for run in date_runs
        ]
    )
)


[1/19] Task7 20260701 -> 20260702
[CACHE HIT] D+1 opening cache 20260701 20260702: basket_minute_wide_20260702_ea54ee2142d4.pkl (3.99 MB)
[2/19] Task7 20260702 -> 20260703
[CACHE HIT] D+1 opening cache 20260702 20260703: basket_minute_wide_20260703_bfa50f229284.pkl (3.99 MB)
[3/19] Task7 20260703 -> 20260706
[CACHE HIT] D+1 opening cache 20260703 20260706: basket_minute_wide_20260706_b0ca6a949f2e.pkl (3.99 MB)
[4/19] Task7 20260706 -> 20260707
[CACHE HIT] D+1 opening cache 20260706 20260707: basket_minute_wide_20260707_14ca0077a818.pkl (3.99 MB)
[5/19] Task7 20260707 -> 20260708
[CACHE HIT] D+1 opening cache 20260707 20260708: basket_minute_wide_20260708_828a77b111aa.pkl (3.99 MB)
[6/19] Task7 20260708 -> 20260709
[CACHE HIT] D+1 opening cache 20260708 20260709: basket_minute_wide_20260709_a7ccd4bdc8b1.pkl (3.99 MB)
[7/19] Task7 20260709 -> 20260710
[CACHE HIT] D+1 opening cache 20260709 20260710: basket_minute_wide_20260710_7c4ba95815ad.pkl (3.99 MB)
[8/19] Task7 20260710 -> 20260713


,construction_date,evaluation_date,unavailable_component_count,manual_not_in_universe
0,20260701,20260702,2,[]
1,20260702,20260703,3,[]
2,20260703,20260706,2,[]
3,20260706,20260707,2,[]
4,20260707,20260708,5,[]
5,20260708,20260709,2,[]
6,20260709,20260710,2,[]
7,20260710,20260713,1,[]
8,20260713,20260714,1,[]
9,20260714,20260715,1,[]


## Task8：逐日在全部成分股中排除不可买后重新拟合 basket3


In [23]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task8 "
        f"{run.construction_date}"
    )
    run.build_basket3()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "basket3_invested_amount": run.basket3[
                    "target_market_value"
                ].sum(),
                "basket3_held_stock_count": int(
                    (run.basket3["target_qty"] > 0).sum()
                ),
                "excluded_component_count": len(
                    run.all_unavailable_codes
                ),
            }
            for run in date_runs
        ]
    )
)


[1/19] Task8 20260701
[pareto basket3] round=1, raw=1,955, local=68, materialized=68, frontier=68, beam=20, new=68, best_amount=0.18098229, best_TE=562.473602%
[pareto basket3] round=5, raw=43,385, local=760, materialized=748, frontier=95, beam=20, new=95, best_amount=0.17433378, best_TE=500.809160%
[pareto basket3] round=10, raw=26,967, local=451, materialized=447, frontier=114, beam=20, new=91, best_amount=0.16941284, best_TE=481.524393%
[pareto basket3] round=15, raw=21,871, local=332, materialized=307, frontier=97, beam=20, new=73, best_amount=0.16661823, best_TE=468.944677%
[pareto basket3] round=20, raw=17,345, local=328, materialized=306, frontier=159, beam=20, new=115, best_amount=0.16400146, best_TE=458.635457%
[pareto basket3] round=25, raw=17,820, local=389, materialized=364, frontier=184, beam=20, new=99, best_amount=0.16324194, best_TE=449.979047%
[pareto basket3] round=30, raw=17,204, local=352, materialized=323, frontier=210, beam=20, new=139, best_amount=0.16321414, bes

,construction_date,basket3_invested_amount,basket3_held_stock_count,excluded_component_count
0,20260701,4522118.70,277,2
1,20260702,4501427.93,277,3
2,20260703,4519745.01,276,2
3,20260706,4522371.80,278,2
4,20260707,4517775.98,276,5
5,20260708,4509839.30,280,2
6,20260709,4506748.11,279,2
7,20260710,4502238.37,277,1
8,20260713,4515602.96,278,1
9,20260714,4502913.39,277,1


## Task9：逐日读取 D+1 指数 09:31 close


In [24]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task9 "
        f"{run.construction_date} -> {run.evaluation_date}"
    )
    run.load_index_opening_price()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "evaluation_date": run.evaluation_date,
                "construction_index_close": (
                    run.construction_index_close
                ),
                "opening_index_close": run.opening_index_close,
            }
            for run in date_runs
        ]
    )
)


[1/19] Task9 20260701 -> 20260702
[2/19] Task9 20260702 -> 20260703
[3/19] Task9 20260703 -> 20260706
[4/19] Task9 20260706 -> 20260707
[5/19] Task9 20260707 -> 20260708
[6/19] Task9 20260708 -> 20260709
[7/19] Task9 20260709 -> 20260710
[8/19] Task9 20260710 -> 20260713
[9/19] Task9 20260713 -> 20260714
[10/19] Task9 20260714 -> 20260715
[11/19] Task9 20260715 -> 20260716
[12/19] Task9 20260716 -> 20260717
[13/19] Task9 20260721 -> 20260722
[14/19] Task9 20260722 -> 20260723
[15/19] Task9 20260723 -> 20260724
[16/19] Task9 20260724 -> 20260727
[17/19] Task9 20260727 -> 20260728
[18/19] Task9 20260730 -> 20260731
[19/19] Task9 20260731 -> 20260803


,construction_date,evaluation_date,construction_index_close,opening_index_close
0,20260701,20260702,4958.9773,4864.983
1,20260702,20260703,4812.2957,4829.190
2,20260703,20260706,4842.1737,4855.503
3,20260706,20260707,4841.9980,4808.960
4,20260707,20260708,4792.2624,4798.470
5,20260708,20260709,4755.5338,4775.219
6,20260709,20260710,4876.3125,4885.546
7,20260710,20260713,4780.7867,4752.512
8,20260713,20260714,4695.3830,4705.830
9,20260714,20260715,4796.5020,4792.022


## Task10：逐日计算 basket1 和 basket3 的 09:31 偏差


In [25]:
task11_summary_by_date = {}
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task10 "
        f"{run.construction_date}"
    )
    task11_summary_by_date[run.construction_date] = run.evaluate()


[1/19] Task10 20260701
[2/19] Task10 20260702
[3/19] Task10 20260703
[4/19] Task10 20260706
[5/19] Task10 20260707
[6/19] Task10 20260708
[7/19] Task10 20260709
[8/19] Task10 20260710
[9/19] Task10 20260713
[10/19] Task10 20260714
[11/19] Task10 20260715
[12/19] Task10 20260716
[13/19] Task10 20260721
[14/19] Task10 20260722
[15/19] Task10 20260723
[16/19] Task10 20260724
[17/19] Task10 20260727
[18/19] Task10 20260730
[19/19] Task10 20260731


## Task11：汇总两个篮子并使用 Plotly 绘图

核心表 `df_deviation_summary_all` 每个构建日固定按 basket1、basket3 输出两行，共 `2 * len(construction_date_ls)` 行、9 列。随后使用 Plotly 绘制 `common_base_gap_pct`、`return_gap_pct`、`cash_adjusted_gap_pct` 三张交互折线图；第一张图的副轴为 `common_base_gap_amount`。


In [26]:
df_deviation_summary_all = combine_task11_summaries(
    task11_summary_by_date,
    construction_date_ls,
)
# 保留原 notebook 常用变量名，现指向批量大表。
df_deviation_summary = df_deviation_summary_all

task11_report_path = (
    batch_reports_dir
    / "all_dates_two_basket_deviation_0931.csv"
)
df_deviation_summary_all.to_csv(
    task11_report_path,
    index=False,
    encoding="utf-8-sig",
)

assert len(df_deviation_summary_all) == 2 * len(
    construction_date_ls
)
assert (
    df_deviation_summary_all.groupby(
        "construction_date", sort=False
    ).size() == 2
).all()

print("Task11 批量报表:", task11_report_path)
display(df_deviation_summary_all)


Task11 批量报表: E:\Codex\系统\Stock-Index-Fitting\data\000300\20260807-165523_limit_impact_v01_batch\04_reports\all_dates_two_basket_deviation_0931.csv


,construction_date,basket,held_stock_count,basket_build_amount,basket_amount_0931,common_base_gap_amount,common_base_gap_pct,return_gap_pct,cash_adjusted_gap_pct
0,20260701,basket1,274,4514954.90,4429744.52,367.754297,0.008303,0.008145,0.008303
1,20260701,basket3,277,4522118.70,4436373.52,6996.754297,0.157963,-0.000691,-0.003771
2,20260702,basket1,278,4514452.07,4530177.75,-122.994388,-0.002715,-0.002724,-0.002715
3,20260702,basket3,277,4501427.93,4517462.81,-12837.934388,-0.283379,0.005152,0.004110
4,20260703,basket1,281,4509257.47,4520331.78,-1338.554411,-0.029603,-0.029685,-0.029603
5,20260703,basket3,276,4519745.01,4531052.11,9381.775589,0.207485,-0.025104,-0.024455
6,20260706,basket1,281,4519512.45,4492109.59,3434.751317,0.076520,0.075998,0.076520
7,20260706,basket3,278,4522371.80,4493889.40,5214.561317,0.116172,0.052511,0.052470
8,20260707,basket1,277,4518819.08,4520311.97,-4360.508454,-0.096372,-0.096497,-0.096372
9,20260707,basket3,276,4517775.98,4518494.85,-6177.628454,-0.136532,-0.113622,-0.113478


In [13]:
required_plot_columns = {
    "construction_date",
    "basket",
    "common_base_gap_pct",
    "common_base_gap_amount",
    "return_gap_pct",
    "cash_adjusted_gap_pct",
}
missing_columns = required_plot_columns - set(
    df_deviation_summary_all.columns
)
if missing_columns:
    raise ValueError(
        f"绘图数据缺少字段: {sorted(missing_columns)}"
    )

basket_order = ["basket1", "basket3"]
basket_colors = {
    "basket1": "#1f77b4",
    "basket3": "#2ca02c",
}

plot_df = df_deviation_summary_all.copy()
plot_df["construction_date"] = pd.to_datetime(
    plot_df["construction_date"].astype(str),
    format="%Y%m%d",
)
plot_df["basket"] = pd.Categorical(
    plot_df["basket"],
    categories=basket_order,
    ordered=True,
)
plot_df = plot_df.sort_values(
    ["construction_date", "basket"]
).reset_index(drop=True)

if plot_df[list(required_plot_columns)].isna().any(axis=None):
    bad_rows = plot_df.loc[
        plot_df[list(required_plot_columns)].isna().any(axis=1)
    ]
    display(bad_rows)
    raise ValueError("绘图数据中存在缺失值")


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_metric_plotly(
    data,
    metric,
    title,
    *,
    secondary_amount=False,
):
    fig = make_subplots(
        specs=[[{"secondary_y": secondary_amount}]]
    )

    # 主轴：basket1 和 basket3 的百分比指标。
    for basket in basket_order:
        basket_data = data.loc[data["basket"] == basket]
        fig.add_trace(
            go.Scatter(
                x=basket_data["construction_date"],
                y=basket_data[metric],
                mode="lines+markers",
                name=f"{basket} - {metric}",
                line={
                    "color": basket_colors[basket],
                    "width": 2.5,
                },
                marker={"size": 7},
                hovertemplate=(
                    "%{x|%Y-%m-%d}"
                    f"<br>{basket}"
                    f"<br>{metric}: %{{y:.6f}}%"
                    "<extra></extra>"
                ),
            ),
            secondary_y=False,
        )

    # 第一张图副轴：两个篮子的 common_base_gap_amount。
    if secondary_amount:
        for basket in basket_order:
            basket_data = data.loc[data["basket"] == basket]
            fig.add_trace(
                go.Scatter(
                    x=basket_data["construction_date"],
                    y=basket_data["common_base_gap_amount"],
                    mode="lines+markers",
                    name=(
                        f"{basket} - common_base_gap_amount"
                    ),
                    line={
                        "color": basket_colors[basket],
                        "width": 1.8,
                        "dash": "dash",
                    },
                    marker={"size": 7, "symbol": "x"},
                    opacity=0.75,
                    hovertemplate=(
                        "%{x|%Y-%m-%d}"
                        f"<br>{basket}"
                        "<br>common_base_gap_amount: "
                        "%{y:,.2f}"
                        "<extra></extra>"
                    ),
                ),
                secondary_y=True,
            )

    fig.add_hline(
        y=0,
        line_width=1,
        line_dash="dot",
        line_color="black",
        secondary_y=False,
    )
    fig.update_layout(
        title=title,
        xaxis_title="Construction Date",
        template="plotly_white",
        hovermode="x unified",
        width=1100,
        height=600,
        legend={
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "left",
            "x": 0,
        },
        margin={"t": 120},
    )
    fig.update_xaxes(
        tickformat="%Y-%m-%d",
        showgrid=True,
    )
    fig.update_yaxes(
        title_text=f"{metric}（%）",
        ticksuffix="%",
        showgrid=True,
        secondary_y=False,
    )
    if secondary_amount:
        fig.update_yaxes(
            title_text="common_base_gap_amount",
            tickformat=",.0f",
            showgrid=False,
            secondary_y=True,
        )

    fig.show()
    return fig


plotly_figures = {
    "common_base_gap": plot_metric_plotly(
        plot_df,
        "common_base_gap_pct",
        "Basket Common Base Gap",
        secondary_amount=True,
    ),
    "return_gap": plot_metric_plotly(
        plot_df,
        "return_gap_pct",
        "Basket Return Gap",
    ),
    "cash_adjusted_gap": plot_metric_plotly(
        plot_df,
        "cash_adjusted_gap_pct",
        "Basket Cash-adjusted Gap",
    ),
}


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_metric_plotly(
    data,
    metric,
    title,
    *,
    secondary_amount=False,
):
    fig = make_subplots(
        specs=[[{"secondary_y": secondary_amount}]]
    )

    # 主轴：basket1 和 basket3 的百分比指标。
    for basket in basket_order:
        basket_data = data.loc[data["basket"] == basket]
        fig.add_trace(
            go.Scatter(
                x=basket_data["construction_date"],
                y=basket_data[metric],
                mode="lines+markers",
                name=f"{basket} - {metric}",
                line={
                    "color": basket_colors[basket],
                    "width": 2.5,
                },
                marker={"size": 7},
                hovertemplate=(
                    "%{x|%Y-%m-%d}"
                    f"<br>{basket}"
                    f"<br>{metric}: %{{y:.6f}}%"
                    "<extra></extra>"
                ),
            ),
            secondary_y=False,
        )

    # 第一张图副轴：两个篮子的 common_base_gap_amount。
    if secondary_amount:
        for basket in basket_order:
            basket_data = data.loc[data["basket"] == basket]
            fig.add_trace(
                go.Scatter(
                    x=basket_data["construction_date"],
                    y=basket_data["common_base_gap_amount"],
                    mode="lines+markers",
                    name=(
                        f"{basket} - common_base_gap_amount"
                    ),
                    line={
                        "color": basket_colors[basket],
                        "width": 1.8,
                        "dash": "dash",
                    },
                    marker={"size": 7, "symbol": "x"},
                    opacity=0.75,
                    hovertemplate=(
                        "%{x|%Y-%m-%d}"
                        f"<br>{basket}"
                        "<br>common_base_gap_amount: "
                        "%{y:,.2f}"
                        "<extra></extra>"
                    ),
                ),
                secondary_y=True,
            )

    fig.add_hline(
        y=0,
        line_width=1,
        line_dash="dot",
        line_color="black",
        secondary_y=False,
    )
    fig.update_layout(
        title=title,
        xaxis_title="Construction Date",
        template="plotly_white",
        hovermode="x unified",
        width=1100,
        height=600,
        legend={
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "left",
            "x": 0,
        },
        margin={"t": 120},
    )
    fig.update_xaxes(
        tickformat="%Y-%m-%d",
        showgrid=True,
    )
    fig.update_yaxes(
        title_text=f"{metric}（%）",
        ticksuffix="%",
        showgrid=True,
        secondary_y=False,
    )
    if secondary_amount:
        fig.update_yaxes(
            title_text="common_base_gap_amount",
            tickformat=",.0f",
            showgrid=False,
            secondary_y=True,
        )

    fig.show()
    return fig


plotly_figures = {
    "common_base_gap": plot_metric_plotly(
        plot_df,
        "common_base_gap_pct",
        "Basket Common Base Gap",
        secondary_amount=True,
    ),
    "return_gap": plot_metric_plotly(
        plot_df,
        "return_gap_pct",
        "Basket Return Gap",
    ),
    "cash_adjusted_gap": plot_metric_plotly(
        plot_df,
        "cash_adjusted_gap_pct",
        "Basket Cash-adjusted Gap",
    ),
}


In [14]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_metric_plotly(
    data,
    metric,
    title,
    *,
    secondary_amount=False,
):
    fig = make_subplots(
        specs=[[{"secondary_y": secondary_amount}]]
    )

    # 主轴：basket1 和 basket3 的百分比指标。
    for basket in basket_order:
        basket_data = data.loc[data["basket"] == basket]
        fig.add_trace(
            go.Scatter(
                x=basket_data["construction_date"],
                y=basket_data[metric],
                mode="lines+markers",
                name=f"{basket} - {metric}",
                line={
                    "color": basket_colors[basket],
                    "width": 2.5,
                },
                marker={"size": 7},
                hovertemplate=(
                    "%{x|%Y-%m-%d}"
                    f"<br>{basket}"
                    f"<br>{metric}: %{{y:.6f}}%"
                    "<extra></extra>"
                ),
            ),
            secondary_y=False,
        )

    # 第一张图副轴：两个篮子的 common_base_gap_amount。
    if secondary_amount:
        for basket in basket_order:
            basket_data = data.loc[data["basket"] == basket]
            fig.add_trace(
                go.Scatter(
                    x=basket_data["construction_date"],
                    y=basket_data["common_base_gap_amount"],
                    mode="lines+markers",
                    name=(
                        f"{basket} - common_base_gap_amount"
                    ),
                    line={
                        "color": basket_colors[basket],
                        "width": 1.8,
                        "dash": "dash",
                    },
                    marker={"size": 7, "symbol": "x"},
                    opacity=0.75,
                    hovertemplate=(
                        "%{x|%Y-%m-%d}"
                        f"<br>{basket}"
                        "<br>common_base_gap_amount: "
                        "%{y:,.2f}"
                        "<extra></extra>"
                    ),
                ),
                secondary_y=True,
            )

    fig.add_hline(
        y=0,
        line_width=1,
        line_dash="dot",
        line_color="black",
        secondary_y=False,
    )
    fig.update_layout(
        title=title,
        xaxis_title="Construction Date",
        template="plotly_white",
        hovermode="x unified",
        width=1100,
        height=600,
        legend={
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "left",
            "x": 0,
        },
        margin={"t": 120},
    )
    fig.update_xaxes(
        tickformat="%Y-%m-%d",
        showgrid=True,
    )
    fig.update_yaxes(
        title_text=f"{metric}（%）",
        ticksuffix="%",
        showgrid=True,
        secondary_y=False,
    )
    if secondary_amount:
        fig.update_yaxes(
            title_text="common_base_gap_amount",
            tickformat=",.0f",
            showgrid=False,
            secondary_y=True,
        )

    fig.show()
    return fig


plotly_figures = {
    "common_base_gap": plot_metric_plotly(
        plot_df,
        "common_base_gap_pct",
        "Basket Common Base Gap",
        secondary_amount=True,
    ),
    "return_gap": plot_metric_plotly(
        plot_df,
        "return_gap_pct",
        "Basket Return Gap",
    ),
    "cash_adjusted_gap": plot_metric_plotly(
        plot_df,
        "cash_adjusted_gap_pct",
        "Basket Cash-adjusted Gap",
    ),
}


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_metric_plotly(
    data,
    metric,
    title,
    *,
    secondary_amount=False,
):
    fig = make_subplots(
        specs=[[{"secondary_y": secondary_amount}]]
    )

    # 主轴：basket1 和 basket3 的百分比指标。
    for basket in basket_order:
        basket_data = data.loc[data["basket"] == basket]
        fig.add_trace(
            go.Scatter(
                x=basket_data["construction_date"],
                y=basket_data[metric],
                mode="lines+markers",
                name=f"{basket} - {metric}",
                line={
                    "color": basket_colors[basket],
                    "width": 2.5,
                },
                marker={"size": 7},
                hovertemplate=(
                    "%{x|%Y-%m-%d}"
                    f"<br>{basket}"
                    f"<br>{metric}: %{{y:.6f}}%"
                    "<extra></extra>"
                ),
            ),
            secondary_y=False,
        )

    # 第一张图副轴：两个篮子的 common_base_gap_amount。
    if secondary_amount:
        for basket in basket_order:
            basket_data = data.loc[data["basket"] == basket]
            fig.add_trace(
                go.Scatter(
                    x=basket_data["construction_date"],
                    y=basket_data["common_base_gap_amount"],
                    mode="lines+markers",
                    name=(
                        f"{basket} - common_base_gap_amount"
                    ),
                    line={
                        "color": basket_colors[basket],
                        "width": 1.8,
                        "dash": "dash",
                    },
                    marker={"size": 7, "symbol": "x"},
                    opacity=0.75,
                    hovertemplate=(
                        "%{x|%Y-%m-%d}"
                        f"<br>{basket}"
                        "<br>common_base_gap_amount: "
                        "%{y:,.2f}"
                        "<extra></extra>"
                    ),
                ),
                secondary_y=True,
            )

    fig.add_hline(
        y=0,
        line_width=1,
        line_dash="dot",
        line_color="black",
        secondary_y=False,
    )
    fig.update_layout(
        title=title,
        xaxis_title="Construction Date",
        template="plotly_white",
        hovermode="x unified",
        width=1100,
        height=600,
        legend={
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "left",
            "x": 0,
        },
        margin={"t": 120},
    )
    fig.update_xaxes(
        tickformat="%Y-%m-%d",
        showgrid=True,
    )
    fig.update_yaxes(
        title_text=f"{metric}（%）",
        ticksuffix="%",
        showgrid=True,
        secondary_y=False,
    )
    if secondary_amount:
        fig.update_yaxes(
            title_text="common_base_gap_amount",
            tickformat=",.0f",
            showgrid=False,
            secondary_y=True,
        )

    fig.show()
    return fig


plotly_figures = {
    "common_base_gap": plot_metric_plotly(
        plot_df,
        "common_base_gap_pct",
        "Basket Common Base Gap",
        secondary_amount=True,
    ),
    "return_gap": plot_metric_plotly(
        plot_df,
        "return_gap_pct",
        "Basket Return Gap",
    ),
    "cash_adjusted_gap": plot_metric_plotly(
        plot_df,
        "cash_adjusted_gap_pct",
        "Basket Cash-adjusted Gap",
    ),
}
